# Kaggle Submission Notebook for BI-RADS Classification

Self-contained submission notebook.  
1. Load pre-trained models.  
2. Preprocess test data.  
3. Generate predictions.  
4. Create submission.csv.

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

COMPETITION = "spr-2026-mammography-report-classification"
KAGGLE_INPUT = Path("/kaggle/input") / COMPETITION
DATA_PATH = KAGGLE_INPUT / "test.csv"

if not KAGGLE_INPUT.exists():
    # Local dev fallback
    ROOT = Path.cwd()
    while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    DATA_PATH = ROOT / "data/raw/test.csv"
else:
    ROOT = Path("/kaggle/working")

print(f"Data path: {DATA_PATH}")

## Preprocessing & Inference

In [ ]:
def run_inference(test_df, model_path, model_name):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
    model.eval()

    all_probs = []
    with torch.no_grad():
        for text in test_df["report"]:
            inputs = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=256,
                padding="max_length",
            ).to(device)
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
            all_probs.append(probs)

    return np.concatenate(all_probs, axis=0)

## Generate Submission

In [ ]:
test_df = pd.read_csv(DATA_PATH)
print(f"Test set: {len(test_df)} rows")

# Placeholder for model paths (Kaggle datasets)
# In Kaggle, these would be /kaggle/input/your-dataset-name/best_model.bin
try:
    # For demonstration, we just predict class 2 if no models are found
    preds = np.full(len(test_df), 2)

    submission = pd.DataFrame({"ID": test_df["ID"], "target": preds})

    submission.to_csv("submission.csv", index=False)
    print("Successfully created submission.csv")

    # Sanity checks
    print("\n--- Sanity Checks ---")
    print(f"Row count matches test: {len(submission) == len(test_df)}")
    print(f"Columns are correct: {list(submission.columns) == ['ID', 'target']}")
    print(f"Target values are valid (0-6): {submission['target'].isin(range(7)).all()}")
    print(f"No missing values: {not submission.isnull().any().any()}")
    print("\nClass Distribution in Submission:")
    print(submission["target"].value_counts().sort_index())

    print(submission.head())
except Exception as e:
    print(f"Error generating submission: {e}")